# Sufficient system: sample from Station 5 only using G⁵

Purely observational — the environments for ICP are the exogenous variables
themselves (no interventions):

* System 1: 87, 82, 89, 83, 84, 86 — E := 87, Y := 84, X := 82, 89, 83
* System 2: 85, 94, 95, 90, 91, 97, 93, 92 — E := 85, 94, Y := 92, U := 95

(88, 96 removed: isolated nodes)

In [1]:
import pandas as pd

from causalAssembly.models_dag import ProductionLineGraph
from causalAssembly.drf_fitting import fit_drf

seed = 2026
n_select = 5000

system1 = ["Station5_mp_" + str(n) for n in [87, 82, 89, 83, 84, 86]]
system2 = ["Station5_mp_" + str(n) for n in [85, 94, 95, 90, 91, 97, 93, 92]]

In [2]:
# load in ground truth and data
assembly_line = ProductionLineGraph.get_ground_truth()
assembly_line_data = ProductionLineGraph.get_data()

In [3]:
# speed-up: fit_drf is slow because rpy2 deep-converts the fitted forest
import rpy2.robjects as ro
from rpy2.robjects.conversion import localconverter
from causalAssembly.drf_fitting import DRF, R_CONVERTER, drf_r_package

def fit(self, X, Y):
    self.X_train, self.Y_train = X, Y
    with localconverter(R_CONVERTER):
        X_r = ro.conversion.py2rpy(X)
        Y_r = ro.conversion.py2rpy(Y)
    self.r_fit_object = drf_r_package.drf(X_r, Y_r, **self.fit_params)

DRF.fit = fit

In [4]:
assembly_line.Station5.drf = fit_drf(assembly_line.Station5, data=assembly_line_data)

In [5]:
# sample from station5 only and split into the two systems
import numpy as np

assembly_line.Station5.random_state = np.random.default_rng(seed)
station5_sample = assembly_line.Station5.sample_from_drf(size=n_select)

station5_sample[system1].to_csv("station5_system1.csv", index=False)
station5_sample[system2].to_csv("station5_system2.csv", index=False)
station5_sample.describe().round(3)

,Station5_mp_82,Station5_mp_83,Station5_mp_84,Station5_mp_85,Station5_mp_86,Station5_mp_87,Station5_mp_88,Station5_mp_89,Station5_mp_90,Station5_mp_91,Station5_mp_92,Station5_mp_93,Station5_mp_94,Station5_mp_95,Station5_mp_96,Station5_mp_97
count,5000.000,5000.000,5000.000,5000.000,5000.000,5000.000,5000.000,5000.000,5000.000,5000.000,5000.000,5000.000,5000.000,5000.000,5000.000,5000.000
mean,11390.317,5965.668,0.005,236.082,0.001,5424.222,4467620.976,0.005,11155.100,5874.614,0.005,382.831,0.001,5279.282,4428728.500,0.005
std,110.715,291.492,0.000,11.930,0.000,389.070,338622.592,0.000,124.656,296.127,0.000,13.065,0.000,399.416,327192.761,0.000
min,10637.470,5158.513,0.005,198.446,0.000,4125.437,3892798.773,0.005,10602.860,5191.198,0.005,339.030,0.000,3927.007,3915512.065,0.004
25%,11336.680,5758.385,0.005,227.286,0.000,5170.825,4198689.618,0.005,11079.520,5662.251,0.005,374.279,0.001,5029.618,4173269.753,0.005
50%,11401.410,5949.370,0.005,237.009,0.001,5445.911,4409772.518,0.005,11164.280,5863.491,0.005,382.610,0.001,5300.064,4360834.542,0.005
75%,11454.600,6173.200,0.005,245.301,0.001,5689.716,4676309.464,0.005,11237.980,6071.298,0.005,390.942,0.001,5553.234,4599639.570,0.005
max,11835.930,6757.530,0.006,267.135,0.002,6661.988,5824053.891,0.005,11722.500,6685.751,0.006,440.290,0.002,6480.265,5668918.326,0.005


## Replication: 50 samples for the seed loop

One fit, many samples — only sampling is repeated.

In [7]:
import os

n_seeds = 50
os.makedirs("seeds", exist_ok=True)

for k in range(1, n_seeds + 1):
    assembly_line.Station5.random_state = np.random.default_rng(k)
    s = assembly_line.Station5.sample_from_drf(size=n_select)
    s[system1].to_csv(f"seeds/station5_system1_seed{k}.csv", index=False)
    s[system2].to_csv(f"seeds/station5_system2_seed{k}.csv", index=False)
    print(k, end=" ")

1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 19 20 21 22 23 24 25 26 27 28 29 30 31 32 33 34 35 36 37 38 39 40 41 42 43 44 45 46 47 48 49 50 